# 02 - Loss simulation

Image → CCSDS 122 → CCSDS 133 packets → channel → depacketize → loss-tolerant decode.

Each `Trial` holds the ground truth, the damaged decode, masks and the coefficient bounds.

In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from hmi_infill import prep, synthetic, viz

FITS_DIR = Path("../data/raw/hmi.M_45s")   # change to the series you downloaded
CROP = None                                 # e.g. 1024 for faster experiments
paths = sorted(FITS_DIR.glob("*.fits"))
if paths:
    loaded = [prep.load_hmi_int(p, crop=CROP) for p in paths[:6]]
    frames, valids, info, header = [l[0] for l in loaded], [l[1] for l in loaded], loaded[0][2], loaded[0][3]
    print(f"{len(frames)} FITS frames from {FITS_DIR}, shape {frames[0].shape}")
else:
    print("No FITS found -> synthetic magnetogram sequence")
    conv = [prep.to_int(f, content="MAGNETOGRAM") for f in synthetic.sequence(6, 512, "magnetogram", seed=0)]
    frames, valids, info, header = [c[0] for c in conv], [c[1] for c in conv], conv[0][2], None
print(info)

In [ ]:
from hmi_infill.ccsds122 import CodecParams
from hmi_infill.channel import Bernoulli, GilbertElliott
from hmi_infill.pipeline import simulate_sequence

codec = CodecParams(segment_blocks=64)
channel = GilbertElliott.from_mean(mean_loss=0.03, mean_burst=4)
trials = simulate_sequence(frames, codec=codec, channel=channel, seed=1, max_packet_len=512, valid=valids)
for t in trials:
    m = t.meta
    print(f"frame {m['frame']}: {m['bits_per_pixel']:.2f} bpp, {m['n_packets']} packets, {m['packet_loss_rate']:.2%} lost -> "
          f"segments lost/trunc {m['segments_lost']}/{m['segments_truncated']}, "
          f"pixels core {m['pixels_core_fraction']:.2%}, affected {m['pixels_lost_fraction']:.2%}")

## Damage of one frame

* **core** (red): 8×8 footprints of lost blocks
* **lost** (cyan): every pixel the inverse DWT links to a lost coefficient
* **block status**: truncated segments still give coarse coefficients

In [ ]:
k = int(np.argmax([t.meta["blocks_lost"] for t in trials]))
t = trials[k]
nrm = viz.norm_for(t.truth.astype(float), symmetric=info.content.upper() != "CONTINUUM INTENSITY")
fig, axs = plt.subplots(1, 3, figsize=(18, 5.5))
viz.show(t.truth, axs[0], f"truth (frame {k})", norm=nrm)
viz.show(t.damaged, axs[1], "damaged + masks", norm=nrm)
viz.overlay_mask(axs[1], t.mask_lost, "cyan", 0.25)
viz.overlay_mask(axs[1], t.mask_core, "red", 0.5)
viz.show_block_status(t.block_status, axs[2])
plt.tight_layout()

## Precision of what arrived

For each block: the lowest bit-plane received (0 = exact, 99 = nothing).

In [ ]:
from hmi_infill.ccsds122.blocks import block_grid
p = t.bitplane[:, 1:].max(axis=1).astype(float)
p[p >= 99] = np.nan
fig, ax = plt.subplots(figsize=(7, 6))
viz.show(p.reshape(block_grid(t.coeff_est.shape)), ax, "worst AC bit-plane received per block", "viridis",
         norm=viz.ImageNormalize(vmin=0, vmax=np.nanmax(p) if np.isfinite(np.nanmax(p)) else 1))

## Zoom on a lost region

In [ ]:
ys, xs = np.nonzero(t.mask_core)
if ys.size:
    cy, cx = int(np.median(ys)), int(np.median(xs))
    h = 64
    y0, x0 = max(cy - h, 0), max(cx - h, 0)
    zoom = (y0, y0 + 2 * h, x0, x0 + 2 * h)
    viz.compare(t.truth, t.damaged, t.damaged, t.mask_lost, zoom=zoom, title="damaged decode (no infill)",
                symmetric=info.content.upper() != "CONTINUUM INTENSITY")

## Packet loss rate vs. damaged area

Independent losses vs. bursts at the same average rate.

In [ ]:
rates = [0.005, 0.01, 0.02, 0.05, 0.1]
res = {"Bernoulli": [], "Gilbert-Elliott, burst 8": []}
for r in rates:
    for name, ch in (("Bernoulli", Bernoulli(r)), ("Gilbert-Elliott, burst 8", GilbertElliott.from_mean(r, 8))):
        ts = simulate_sequence(frames[:2], codec=codec, channel=ch, seed=2, max_packet_len=512, valid=valids[:2])
        res[name].append(np.mean([x.meta["pixels_core_fraction"] for x in ts]))
plt.figure(figsize=(7, 4.5))
for name, v in res.items():
    plt.plot(np.array(rates) * 100, np.array(v) * 100, "o-", label=name)
plt.xlabel("packet loss rate [%]"); plt.ylabel("pixels in lost blocks [%]"); plt.legend(); plt.grid(alpha=.3)

## Save trials for the infill experiments

In [ ]:
out = Path("../trials/notebook_demo")
for t in trials:
    t.save(out / f"frame_{t.meta['frame']:02d}")
print("saved to", out.resolve())